# 6. A challenger model

The scorecard is simple on purpose. It bins every feature and adds up points. This notebook asks what that simplicity costs. A machine-learning model, LightGBM, is fitted on the same train sets and tested on the same test sets. It uses the exact values with no bins, and it can combine features, for example a low FICO score together with a high dti.

If the challenger is only a little better than the scorecard, the scorecard loses little by being simple. If it is much better, the scorecard is leaving information unused.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent
PROCESSED = ROOT / "data" / "processed"

train_36 = pd.read_parquet(PROCESSED / "train_36.parquet")
test_36 = pd.read_parquet(PROCESSED / "test_36.parquet")
train_60 = pd.read_parquet(PROCESSED / "train_60.parquet")
test_60 = pd.read_parquet(PROCESSED / "test_60.parquet")

features = [c for c in train_36.columns if c != "target"]
categories = [c for c in features if not pd.api.types.is_numeric_dtype(train_36[c])]

### 6.1 Preparing the data

LightGBM works on the raw values. Number features go in as they are, and missing values are handled by the model itself. The category features are marked as categories, with the same list of categories in the train and test sets.

In [2]:
def prepare(train, test):
    train, test = train[features].copy(), test[features].copy()
    for c in categories:
        levels = sorted(train[c].dropna().unique())
        train[c] = pd.Categorical(train[c], categories=levels)
        test[c] = pd.Categorical(test[c], categories=levels)
    return train, test


X_train_36, X_test_36 = prepare(train_36, test_36)
X_train_60, X_test_60 = prepare(train_60, test_60)

### 6.2 Fitting the challenger

LightGBM builds many small decision trees, one after another, each correcting the mistakes of the ones before. It is given all 16 features and decides for itself which ones to use. The settings keep the trees small, so the model cannot simply memorise the train set.

In [3]:
def fit_lightgbm(X, y):
    model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15,
                               min_child_samples=200, subsample=0.8, subsample_freq=1,
                               colsample_bytree=0.8, random_state=0, verbose=-1)
    return model.fit(X, y)


lgb_36 = fit_lightgbm(X_train_36, train_36["target"])
lgb_60 = fit_lightgbm(X_train_60, train_60["target"])

lgb_pd_36 = lgb_36.predict_proba(X_test_36)[:, 1]
lgb_pd_60 = lgb_60.predict_proba(X_test_60)[:, 1]

### 6.3 Challenger against scorecard

Both models are tested on the same test loans. The Gini shows how well each ranks borrowers, and the average PD, next to the default rate, shows whether each gets the overall level of risk right.

In [5]:
from optbinning import BinningProcess

binning_36 = BinningProcess.load(str(PROCESSED / "binning_36.pkl"))
binning_60 = BinningProcess.load(str(PROCESSED / "binning_60.pkl"))
weights_36 = pd.read_json(PROCESSED / "weights_36.json", typ="series")
weights_60 = pd.read_json(PROCESSED / "weights_60.json", typ="series")


def predict_pd(loans, binning, weights):
    woe = binning.transform(loans[features], metric="woe")
    chosen = [c for c in weights.index if c != "const"]
    score = weights["const"] + woe[chosen] @ weights[chosen]
    return 1 / (1 + np.exp(-score))


card_pd_36 = predict_pd(test_36, binning_36, weights_36)
card_pd_60 = predict_pd(test_60, binning_60, weights_60)

In [6]:
rows = []
for group, test, card_pd, lgb_pd in [("36-month", test_36, card_pd_36, lgb_pd_36),
                                     ("60-month", test_60, card_pd_60, lgb_pd_60)]:
    for model, p in [("Scorecard", card_pd), ("LightGBM", lgb_pd)]:
        auc = roc_auc_score(test["target"], p)
        rows.append({
            "group": group,
            "model": model,
            "AUC": round(auc, 3),
            "Gini": round(2 * auc - 1, 3),
            "average PD": round(p.mean(), 3),
            "default rate": round(test["target"].mean(), 3),
        })

pd.DataFrame(rows)

,group,model,AUC,Gini,average PD,default rate
0,36-month,Scorecard,0.654,0.307,0.134,0.149
1,36-month,LightGBM,0.670,0.341,0.135,0.149
2,60-month,Scorecard,0.607,0.214,0.265,0.248
3,60-month,LightGBM,0.612,0.224,0.263,0.248


LightGBM ranks borrowers a little better than the scorecard. For the 36-month loans its Gini is 0.341 against 0.307, and for the 60-month loans 0.224 against 0.214.

So the scorecard gives up little by being simple. In return, it can be read, explained and checked, which LightGBM cannot.